<!-- nav -->
[← 7 · One constraint, many languages](07_One_Constraint_Many_Languages.ipynb) · [Home](../../README.md) · [Equivalence of the implementations →](../../docs/EQUIVALENCE.md)

# 8 · Constraints for firmware and hardware

**The problem.** The logger's firmware, in C, checks each packet before sending it, and its FPGA's verification
engineers check the same condition with SystemVerilog assertions. In C, `uint8_t + uint8_t` is an `int`, and signed
overflow is undefined. In SystemVerilog, a bit can be `x`, unknown, or `z`, undriven. A constraint that is right in
TypeScript can be wrong in either.

The Ccpp and SystemVerilog dialects model those languages' rules exactly, so you can translate a packet's constraint,
written once in Basic, and evaluate it the way the firmware and the hardware will. A Basic term carries no width: its
ints are unbounded, and what C's and SystemVerilog's types make of a value is theirs. Giving Basic's terms domains,
`int16` and `uint8`, waits for type inference, a later capability.

In [1]:
import { Evaluators, Expressions as E } from "@mbse/expressions";
import * as Ccpp from "@mbse/expressions/Dialects/Ccpp";
import * as SystemVerilog from "@mbse/expressions/Dialects/SystemVerilog";
import * as Translators from "@mbse/expressions/Translators";
import { cold_chain } from "./toolkit.js";

const store = cold_chain(); // with Ccpp's and SystemVerilog's kinds installed
const [Expression, Term] = [store.Expressions.Framework.Expression, store.Expressions.Framework.Term];
const [r, s] = [E.variable("raw"), E.variable("status")];
const packet_ok = store.transaction(() => Expression((e: any) => e.root(r.ge(200n).and_(r.le(800n)).and_(s.bitand(0b100n).eq(0n)))));
const to_c = Translators.between(E.DIALECT, Ccpp.Expressions.DIALECT);
const to_sv = Translators.between(E.DIALECT, SystemVerilog.Expressions.DIALECT);

## Step 1: C

A translation is a writer of the target's term, written in a transaction like any other:

In [2]:
const c = store.transaction(() => Term(to_c.forward(packet_ok)));
console.log(Ccpp.Text.ToText(c));
for (const status of [0n, 0b100n]) {
  console.log(String(status), Ccpp.Evaluators.OfAny(c, { raw: 523n, status }));
}

raw >= 200 && raw <= 800 && (status & 4) == 0


0 true


4 false


## Step 2: where C differs

C's integers have widths: an `int` is 32 bits. Basic's are unbounded, so a sum Basic computes exactly can overflow in
C, and signed overflow is undefined behavior in C: a compiler may assume it never happens. The evaluator throws instead
of picking a result, which is what a test should do:

In [3]:
const increment = store.transaction(() => Term(r.add(1n)));
const increment_c = store.transaction(() => Term(to_c.forward(increment)));
console.log(Ccpp.Text.ToText(increment_c), "in Basic:", String(Evaluators.OfAny(increment, { raw: 2147483647n })));
try {
  Ccpp.Evaluators.OfAny(increment_c, { raw: 2147483647n });
} catch (error) {
  console.log("C:", (error as Error).message);
}

raw + 1 in Basic: 2147483648


C: signed overflow in +: 2147483648 does not fit 'int'


C's own types are C's: a `uint8_t` is promoted to `int` before it is added, so the sum is 300, not 44. Basic has no
counterpart for a constant of its own type yet, since its domain waits for type inference, and the translator refuses
it, naming it:

In [4]:
const C = Ccpp.Expressions;
const total = store.transaction(() => Term(C.binary("+", C.constant(200n, "uint8_t"), C.constant(100n, "uint8_t"))));
console.log(Ccpp.Text.ToText(total), "=", String(Ccpp.Evaluators.OfAny(total, {})));
try {
  to_c.backward(total);
} catch (error) {
  console.log((error as Error).message);
}

(uint8_t)200 + (uint8_t)100 = 300


Ccpp constant 200 of uint8_t has no Basic counterpart


## Step 3: SystemVerilog

SystemVerilog's integers are 4-state vectors. Its evaluator gives a 4-state value, written as SystemVerilog would
display it:

In [5]:
const sv = store.transaction(() => Term(to_sv.forward(packet_ok)));
console.log(SystemVerilog.Text.ToText(sv));
console.log(String(SystemVerilog.Evaluators.OfAny(sv, { raw: 523n, status: 0n })));

raw >= 200 && raw <= 800 && (status & 4) == 0


1'b1


## Step 4: x and z

In simulation, a register that was never reset holds `x`. Whether that matters depends on the bit: the mask
looks only at bit 2, so an `x` elsewhere doesn't change the answer, but an `x` in bit 2 makes the whole condition `x`,
which an assertion reports as a failure to know. `Logic.of` makes a 4-state value from its bits:

In [6]:
const Logic = SystemVerilog.Domains.Logic;
for (const bits of ["00000000", "00000100", "000000x0", "00000x00"]) {
  console.log(bits, String(SystemVerilog.Evaluators.OfAny(sv, { raw: 523n, status: Logic.of(bits) })));
}

00000000 1'b1


00000100 1'b0


000000x0 1'b1


00000x00 1'bx


This is Kleene's unknown again, in hardware's terms: `x` where the data doesn't decide.

## Step 5: what has no counterpart

`has` asks whether a property is present. A C struct or a SystemVerilog register always has its fields, so there's
nothing to translate it to, and the translator refuses, naming the term:

In [7]:
const reported = store.transaction(() => Term(E.variable("shipment").has("celsius")));
for (const translator of [to_c, to_sv]) {
  try {
    translator.forward(reported);
  } catch (error) {
    console.log((error as Error).message);
  }
}

Basic call 'has' has no Ccpp counterpart


Basic call 'has' has no SystemVerilog counterpart


## Why it works this way

- **Model the language, don't approximate it.** Firmware and hardware bugs live in exactly the corners where languages
  differ: promotions, wrapping, undefined behavior, `x`. An evaluator that follows C's or IEEE 1800's rules finds them
  in a test, before the hardware does.
- **The neutral constraint stays neutral.** Basic doesn't take C's promotions or SystemVerilog's sizing. The translator
  writes them out, and the target's evaluator applies them.
- **Refuse what can't be said.** A constraint about whether data is present has no meaning for a fixed-layout packet; a
  silent approximation would be a bug in the translation.

That's the series: a constraint written once, as data, checked, stored, analyzed, specialized, and checked in each
language its users work in.

<!-- nav -->
[← 7 · One constraint, many languages](07_One_Constraint_Many_Languages.ipynb) · [Home](../../README.md) · [Equivalence of the implementations →](../../docs/EQUIVALENCE.md)